# Orientation contrast and the pinned boundary states

Two things, both built on the graded fit used in FIG4, the multi-energy ``logY``
polish (``load_bookended_multienergy_logY_polish()``), which fits 283.7, 284.2,
285.1, 287 and 289 eV jointly with one shared film and per-energy instrument
parameters.

1. The orientation contrast of the ZnPc molecular tensor, $n_{xx}$ against $n_{zz}$,
   over the C K edge.
2. The reflectivity series for each of the four boundary values of the book-ended
   film, varied one at a time with every other parameter held at the published
   value. At the Si substrate these are the pinned orientation $\gamma_\mathrm{Si}$
   (``alpha_si``) and density $\rho_\mathrm{Si}$ (``density_si``); at the free
   surface, $\gamma_\mathrm{vac}$ (``alpha_vac``) and $\rho_\mathrm{vac}$
   (``density_vac``). Each relaxes to the bulk value over $\tau_\mathrm{Si}$ or
   $\tau_\mathrm{vac}$, which stay fixed.

Every energy shown is one of the five fitted energies. Only 283.7 eV is measured
beyond $q \approx 0.074$, so at the other four the high-$q$ part of each curve is
the fitted model extrapolated past the data.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from utils import read_xrr
from utils.graded_objective import (
    extract_graded_fit_context,
    load_bookended_multienergy_logY_polish,
    resolve_data_key,
)
from utils.helpers.fitting_helper import rxr
from utils.helpers.plotting_helper import set_plotting_defaults
from utils.models import configure_refloxide_fitting

configure_refloxide_fitting()
set_plotting_defaults()

E_RES = (283.7, 284.2, 285.1, 287.0, 289.0)   # resonant series of the data
POLS = ("s", "p")
N_Q = 1500
FIG_DPI = 200

data = read_xrr("reflectivity_data", material="znpc", source="hub")
graded_polish = load_bookended_multienergy_logY_polish()   # the FIG4 graded fit
film_fit = graded_polish.film
E_SWATH = tuple(sorted(graded_polish.objectives))          # the five fitted energies

ref_ds = data[resolve_data_key(data, 283.7)]
Q_MAX = float(max(np.max(ref_ds.s.x), np.max(ref_ds.p.x)))
Q = np.linspace(1e-3, Q_MAX, N_Q)
print(f"graded polish from {graded_polish.path.name}, energies {E_SWATH}; q to {Q_MAX:.3f} 1/A")

## 1. Orientation contrast of the molecular tensor, $n_{xx}$ against $n_{zz}$

Molecular-frame optical constants from the DFT tensor (``dft.csv``) and the
experimentally anchored tensor (``exp.csv``). The columns ``n_xx``, ``n_zz`` hold
$\delta$ and ``n_ixx``, ``n_izz`` hold $\beta$, with $n = 1 - \delta + i\beta$. The
$\pi^*$ absorption sits on $\beta_{zz}$, so $z$ is the macrocycle normal and the
transition-dipole axis, $n_\parallel = n_{zz}$ and $n_\perp = n_{xx}$.

**Orientational self-contrast.** For an ensemble whose transition dipoles make the
polar angle $\gamma$ with the film normal, with order parameter $S$, the lab tensor is
$\epsilon_{ij} = \bar\epsilon\,\delta_{ij} + S\,\Delta\epsilon\,(\hat d_i \hat d_j - \delta_{ij}/3)$,
with $\bar\epsilon = (\epsilon_\parallel + 2\epsilon_\perp)/3$ and
$\Delta\epsilon = \epsilon_\parallel - \epsilon_\perp \simeq 2(n_\parallel - n_\perp)$.
Averaged about the normal this gives
$\epsilon_{zz} = \bar\epsilon + S\Delta\epsilon(\cos^2\gamma - 1/3)$ and
$\epsilon_{xx} = \bar\epsilon + S\Delta\epsilon(\tfrac12\sin^2\gamma - 1/3)$.
Two regions of the same material that differ only in orientation differ by
$\Delta n_\mathrm{eff} = (n_\parallel - n_\perp)\,\Delta(S\,P)$, where $P$ is the
geometric weight of the field component probed. The energy dependence of pure
orientation contrast is therefore carried entirely by
$|n_\parallel - n_\perp|^2 = (\delta_\parallel-\delta_\perp)^2 + (\beta_\parallel-\beta_\perp)^2$,
the self-contrast of RSoXS. This follows the standard uniaxial treatment and
should be checked against the equations of *How to RSoXS* (J. Chem. Phys. 163,
061501, 2025) and *Resonant Soft X-ray Scattering for Organic Photovoltaics*
(J. Phys. Chem. B 129, 3529, 2025).

**Homodyne against heterodyne.** In RSoXS the orientational scattering is detected
on its own, so the signal scales as $|\Delta n|^2$. In specular reflectivity the
change of the pinned or surface orientation alters an amplitude that interferes
with the much larger reflection of the rest of the stack, so to first order
$\Delta\ln R$ scales as $|\Delta n|$ times a phase factor. Both are plotted. The
energies of maximum contrast coincide, but their relative weights differ, and the
heterodyne weight falls off far more slowly away from the resonance peaks.

Measures shown.

* $|\Delta n|^2$, the RSoXS self-contrast, and $|\Delta n|$, the reflectivity weight.
* $\Delta\delta$ and $\Delta\beta$ separately, the dispersive and absorptive parts.
* $|\Delta n|/|1-n_\mathrm{iso}|$, the anisotropy relative to the isotropic contrast
  against vacuum, $n_\mathrm{iso} = (2n_{xx}+n_{zz})/3$. Where the isotropic film is
  nearly index matched to vacuum, orientation dominates the reflectivity even though
  $|\Delta n|$ is not at its peak. This is the measure to read for the question about
  283.7 eV.
* $|1-n_{zz}|/|1-n_{xx}|$, the ratio of the two component contrasts against vacuum,
  which diverges where one component is index matched.

The fitted ``energy_offset`` (a few meV) is ignored at this energy spacing.

In [ ]:
from scipy.signal import argrelextrema

from utils import read_ooc

CHECK_E = E_RES
WINDOW = (280.0, 295.0)


def tensor_table(name):
    t = read_ooc(name, material="znpc", source="local")
    t = t[(t.energy >= WINDOW[0]) & (t.energy <= WINDOW[1])]
    t = t.drop_duplicates("energy").sort_values("energy").reset_index(drop=True)
    t["d_delta"] = t.n_xx - t.n_zz
    t["d_beta"] = t.n_ixx - t.n_izz
    t["abs_dn"] = np.hypot(t.d_delta, t.d_beta)
    t["self_contrast"] = t.abs_dn**2   # RSoXS orientational self-contrast
    t["zz_over_xx"] = np.hypot(t.n_zz, t.n_izz) / np.hypot(t.n_xx, t.n_ixx)
    t["iso_contrast"] = np.hypot((2 * t.n_xx + t.n_zz) / 3, (2 * t.n_ixx + t.n_izz) / 3)
    t["rel_aniso"] = t.abs_dn / t.iso_contrast
    return t


def extrema(t, col, order=10):
    y = np.abs(t[col].to_numpy())
    idx = argrelextrema(y, np.greater, order=order)[0]
    return pd.DataFrame({"measure": col, "E (eV)": t.energy.to_numpy()[idx],
                         "|value|": y[idx]}).sort_values("|value|", ascending=False)


def zero_crossings(t, col):
    y, e = t[col].to_numpy(), t.energy.to_numpy()
    k = np.where(np.diff(np.sign(y)) != 0)[0]
    return [float(e[i] - y[i] * (e[i + 1] - e[i]) / (y[i + 1] - y[i])) for i in k]


TENSORS = {n: tensor_table(n) for n in ("dft.csv", "exp.csv")}
for name, t in TENSORS.items():
    print(f"== {name}")
    ex = pd.concat([extrema(t, c) for c in ("self_contrast", "abs_dn", "d_delta", "d_beta", "rel_aniso", "zz_over_xx")])
    display(ex.groupby("measure", sort=False).head(3).round(5))
    at = t.iloc[[int(np.argmin(np.abs(t.energy - e))) for e in CHECK_E]]
    display(at[["energy", "n_xx", "n_zz", "n_ixx", "n_izz", "d_delta", "d_beta",
                "abs_dn", "iso_contrast", "rel_aniso", "zz_over_xx"]].round(6))
    print("self-contrast |dn|^2 at the resonant series, relative to its maximum:",
          {e: round(float(at.self_contrast.iloc[i] / t.self_contrast.max()), 3)
           for i, e in enumerate(CHECK_E)})
    print("delta_xx zero crossings (eV):", np.round(zero_crossings(t, "n_xx"), 2))
    print("delta_zz zero crossings (eV):", np.round(zero_crossings(t, "n_zz"), 2))

fig, axes = plt.subplots(5, 1, figsize=(3.5, 9.2), sharex=True, constrained_layout=True)
t = TENSORS["dft.csv"]
te = TENSORS["exp.csv"]
axes[0].plot(t.energy, t.n_xx, color="C0", lw=0.9, label=r"$\delta_{xx}$")
axes[0].plot(t.energy, t.n_zz, color="C3", lw=0.9, label=r"$\delta_{zz}$")
axes[0].plot(t.energy, t.n_ixx, color="C0", ls="--", lw=0.9, label=r"$\beta_{xx}$")
axes[0].plot(t.energy, t.n_izz, color="C3", ls="--", lw=0.9, label=r"$\beta_{zz}$")
axes[0].axhline(0, color="k", lw=0.4)
axes[0].legend(fontsize=6, frameon=False, ncols=2)
axes[0].set_ylabel("optical constant")
axes[1].plot(t.energy, t.d_delta, color="C2", lw=0.9, label=r"$\Delta\delta$")
axes[1].plot(t.energy, t.d_beta, color="C4", lw=0.9, label=r"$\Delta\beta$")
axes[1].plot(t.energy, t.abs_dn, color="k", lw=1.1, label=r"$|\Delta n|$ DFT")
axes[1].plot(te.energy, te.abs_dn, color="0.5", ls=":", lw=1.0, label=r"$|\Delta n|$ exp")
axes[1].axhline(0, color="k", lw=0.4)
axes[1].legend(fontsize=6, frameon=False, ncols=2)
axes[1].set_ylabel(r"$n_{xx}-n_{zz}$")
axes[2].plot(t.energy, t.self_contrast / t.self_contrast.max(), color="k", lw=1.0,
             label=r"$|\Delta n|^2$, RSoXS (homodyne)")
axes[2].plot(t.energy, t.abs_dn / t.abs_dn.max(), color="C1", lw=1.0,
             label=r"$|\Delta n|$, reflectivity (heterodyne)")
axes[2].legend(fontsize=6, frameon=False)
axes[2].set_ylabel("normalised contrast")
axes[3].semilogy(t.energy, t.rel_aniso, color="k", lw=0.9)
axes[3].set_ylabel(r"$|\Delta n|/|1-n_\mathrm{iso}|$")
axes[4].semilogy(t.energy, t.zz_over_xx, color="k", lw=0.9)
axes[4].set_ylabel(r"$|1-n_{zz}|/|1-n_{xx}|$")
axes[4].set_xlabel("E (eV)")
for ax in axes:
    for e in E_RES:
        ax.axvline(e, color="0.6", lw=0.6, ls="--")
    ax.set_xlim(*WINDOW)
axes[0].set_title("dashed lines at the resonant series " + ", ".join(f"{e:g}" for e in E_RES) + " eV", fontsize=6)
plt.show()

## 2. Reflectivity series for the pinned boundary states

$Rq^4$ on the energy swath for each boundary value, s and p. The colour bar marks
the published value of each parameter with a black tick. The dotted line in a
panel is the measured $q_\mathrm{max}$ at that energy, where it falls inside the
plotted range.

A density change scales the whole tensor, so its spectral weight follows
$|1-n_\mathrm{iso}|$ and s and p respond together. An orientation change scales only
$n_\parallel - n_\perp$ (section 1), with opposite signs in $xx$ and $zz$, so it
separates s from p.

In [ ]:
def film_of(obj):
    return extract_graded_fit_context(obj).film


def data_of(e):
    try:
        return data[resolve_data_key(data, e)]
    except KeyError:
        return None


def qmax_of(e):
    ds = data_of(e)
    return None if ds is None else float(max(np.max(ds.s.x), np.max(ds.p.x)))


ANGLES = np.arange(0.0, 91.0, 15.0)
DENSITIES = np.linspace(0.4, 2.2, 7)
BOUNDARY = {
    "alpha_si": dict(title="substrate orientation", label=r"$\gamma_\mathrm{Si}$ (deg)",
                     values=ANGLES, to_model=np.radians, from_model=np.degrees,
                     cmap="viridis"),
    "density_si": dict(title="substrate density", label=r"$\rho_\mathrm{Si}$ (g cm$^{-3}$)",
                       values=DENSITIES, to_model=lambda v: v, from_model=lambda v: v,
                       cmap="cividis"),
    "alpha_vac": dict(title="surface orientation", label=r"$\gamma_\mathrm{vac}$ (deg)",
                      values=ANGLES, to_model=np.radians, from_model=np.degrees,
                      cmap="viridis"),
    "density_vac": dict(title="surface density", label=r"$\rho_\mathrm{vac}$ (g cm$^{-3}$)",
                        values=DENSITIES, to_model=lambda v: v, from_model=lambda v: v,
                        cmap="cividis"),
}
for name, spec in BOUNDARY.items():
    spec["fitted"] = float(spec["from_model"](float(getattr(film_fit, name).value)))
    print(f"{name:12s} published {spec['fitted']:.3f}")

def obj_at(e):
    """Objective of the multi-energy polish at energy e. Only fitted energies are allowed,
    because ``.at`` silently returns the nearest energy otherwise. The film parameters
    are shared across energies, so every knob change is undone before returning."""
    if float(e) not in graded_polish.objectives:
        raise KeyError(f"{e} eV is not a fitted energy {sorted(graded_polish.objectives)}")
    return graded_polish.at(e)


def sweep(e, name):
    """R(Q) for each value of boundary parameter ``name``, restoring the published value."""
    spec = BOUNDARY[name]
    obj = obj_at(e)
    p = getattr(film_of(obj), name)
    v0 = float(p.value)
    out = {}
    try:
        for pol in POLS:
            rows = []
            for v in spec["values"]:
                p.value = float(spec["to_model"](v))
                rows.append(rxr(Q, obj.model, pol))
            out[pol] = np.array(rows)
    finally:
        p.value = v0
    return out


BSIM = {name: {e: sweep(e, name) for e in E_SWATH} for name in BOUNDARY}
for name in BOUNDARY:
    assert np.ptp(np.log(BSIM[name][283.7]["p"]), axis=0).max() > 1e-6, (
        f"{name} changes did not reach the reflectivity")

In [ ]:
for name, spec in BOUNDARY.items():
    cm = plt.get_cmap(spec["cmap"])
    nrm = plt.Normalize(spec["values"].min(), spec["values"].max())
    colors = cm(np.linspace(0, 1, len(spec["values"])))
    fig, axes = plt.subplots(len(E_SWATH), 2, figsize=(7.2, 1.25 * len(E_SWATH)),
                             sharex=True, constrained_layout=True, dpi=FIG_DPI)
    for ax, pol in zip(axes[0], POLS):
        ax.set_title(f"{pol}-pol", fontsize=8)
    for ax in axes[-1]:
        ax.set_xlabel(r"$q\ (\mathrm{\AA}^{-1})$")
    for row, e in zip(axes, E_SWATH):
        qd = qmax_of(e)
        for ax, pol in zip(row, POLS):
            for c, r in zip(colors, BSIM[name][e][pol]):
                ax.semilogy(Q, r * Q**4, color=c, lw=0.7)
            if qd is not None and qd < Q_MAX:
                ax.axvline(qd, color="0.5", ls=":", lw=0.6)
            ax.text(0.98, 0.9, f"{e:g} eV", transform=ax.transAxes, fontsize=7,
                    va="top", ha="right")
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=nrm, cmap=cm), ax=axes, shrink=0.3,
                      label=spec["label"])
    cb.ax.axhline(spec["fitted"], color="k", lw=1.5)
    fig.suptitle(f"$Rq^4$, {spec['title']}", fontsize=8)
    plt.show()

## 3. Sensitivity figure, the four knobs over the resonant series with a bar-chart summary

Panels are lettered row by row, p-polarised on top, (a) to (e), and s-polarised below,
(f) to (j), with the polarisation also named in the left margin. Panels (a) to (d) and
(f) to (i) are $\gamma_\mathrm{Si}$, $\gamma_\mathrm{vac}$, $\rho_\mathrm{Si}$ and
$\rho_\mathrm{vac}$, each swept alone (colour bar above, published value ticked) with
everything else at the FIG4 polish values.

* *Upper axis of each panel.* 283.7 eV (named once, on the left), the energy measured to high $q$, over its full
  range and on an absolute $Rq^4$ scale.
* *Lower axis.* 284.2, 285.1, 287 and 289 eV, each cut to its measured $q$ range and
  stacked. The stacking is computed separately for every panel: an energy sits as
  high as it can while the envelope of all its swept curves stays ``GAP_DEC``
  decades below the envelope of the energy above, at every shared $q \ge$
  ``Q_YMIN``, so no two traces touch. The order is the same in every panel, so the energies are
  labelled once, in (a) and (f).
* *Panels (e) and (j), Fisher information.* For one dataset $b = (E, \mathrm{pol})$ with
  measured points $q_i$ and log-reflectivity errors $\sigma_i = \Delta R_i / R_i$,
  $$F^{(b)}_{jk} = \sum_i \frac{1}{\sigma_{\mathrm{eff},i}^2}
  \frac{\partial \ln R_i}{\partial \theta_j}\frac{\partial \ln R_i}{\partial \theta_k},$$
  over every parameter varied in the polish plus the four boundary values. The noise is the
  measured error bar plus a fractional error floor per dataset, added in quadrature in
  $\ln R$, $\sigma_{\mathrm{eff},i}^2 = \sigma_i^2 + f_E^2$, with $f_E$ set so that the
  dataset's reduced $\chi^2$ is 1 at the published solution (``FISHER_SCALE =
  "error_model"``). The floor absorbs the model error that the measured error bars
  leave out. ``FISHER_SCALE = None`` gives the predicted sensitivity with the model
  taken as exact.
  From it, the Cramér–Rao bound of dataset $b$ on boundary value $\theta$ is
  $\sigma_\mathrm{CR}^{(b)} = [F^{(b)}_{\theta\theta}]^{-1/2}$, the smallest standard
  error that dataset alone could give with every other parameter known. Each bar is
  the detectability $|\Delta\theta| / \sigma_\mathrm{CR}^{(b)}$, with
  $\Delta\theta = \theta_\mathrm{pin} - \theta_\mathrm{bulk}$, the number of those standard
  errors that separate the pinned value from the bulk value it relaxes to. It is
  dimensionless, so the four knobs share one axis, and it is local, a contrast over the
  curvature at the published values, not a likelihood-ratio test over the full step. The
  bars hold the other parameters fixed. The correlated uncertainty with every
  parameter free and all ten datasets combined, $[F^{-1}]^{1/2}_{\theta\theta}$ with
  $F = \sum_b F^{(b)}$, is printed with the table below for the caption.


In [ ]:
E_PANEL = (283.7, 284.2, 285.1, 287.0, 289.0)
E_HIGHQ = 283.7                      # the one energy measured to high q, drawn on its own axis
Q_YMIN = 0.02                        # ignore the pre-edge dip below this q for the stacking
GAP_DEC = 0.1                        # clearance between neighbouring energies at every q, in decades
FISHER_SCALE = "error_model"   # noise model for the Fisher information:
#   "error_model"  measured error bars plus a fractional error floor f_E per dataset, added in
#                  quadrature in ln R and set so that dataset's reduced chi^2 is 1 at the fit
#   None           measured error bars only, the model taken as exact (predicted sensitivity)
#   "dataset" / "global"  error bars only, information divided by max(chi2_nu, 1)
BULK = {"alpha_si": "alpha_bulk", "alpha_vac": "alpha_bulk",
        "density_si": "density_bulk", "density_vac": "density_bulk"}   # what each boundary relaxes to
ORDER = ("alpha_si", "alpha_vac", "density_si", "density_vac")
SHORT = {"alpha_si": r"$\gamma_\mathrm{Si}$", "alpha_vac": r"$\gamma_\mathrm{vac}$",
         "density_si": r"$\rho_\mathrm{Si}$", "density_vac": r"$\rho_\mathrm{vac}$"}
ROW_POL = ("p", "s")
E_LOWQ = tuple(e for e in E_PANEL if e != E_HIGHQ)

missing = [e for e in E_PANEL if e not in E_SWATH]
assert not missing, f"E_PANEL energies {missing} are not fitted energies {E_SWATH}"
QMEAS = {e: (qmax_of(e) or Q_MAX) for e in E_PANEL}    # measured q range of each energy


def _bounds(p):
    b = getattr(p, "bounds", None)
    lb, ub = getattr(b, "lb", None), getattr(b, "ub", None)
    return (-np.inf if lb is None else float(lb)), (np.inf if ub is None else float(ub))


def _step(p):
    v = float(p.value)
    h = 1e-3 * abs(v) if v != 0 else 1e-6
    lb, ub = _bounds(p)
    return min(h, 1e-3 * (ub - lb)) if np.isfinite(ub - lb) else h


def _data_points(e, pol):
    d = getattr(obj_at(e).data, pol)
    x, y, er = (np.asarray(v, dtype=float) for v in (d.x, d.y, d.y_err))
    m = (y > 0) & (er > 0)
    return x[m], y[m], er[m]


# Fisher information. One block per measured dataset b = (E, pol), evaluated at the
# measured q of that dataset, over every parameter varied in the polish plus the four
# knobs. Rows of J are d lnR / d theta weighted by 1 / sigma_lnR = y / y_err.
#
# Each energy's objective carries its own film object, so a film parameter is one
# physical parameter held by five Python objects. Moving only the 283.7 eV copy leaves
# the other energies untouched (zero derivative, zero information). Film parameters are
# therefore grouped by name and every copy is moved together; instrument parameters stay
# per energy.
FILM_NAMES = ORDER + ("alpha_bulk", "density_bulk", "tau_si", "tau_vac", "total_thick",
                      "surface_roughness")
FILMS = [film_of(obj_at(e)) for e in E_SWATH]
FILM = FILMS[0]


def copies(name):
    out = []
    for f in FILMS:
        q = getattr(f, name, None)
        if q is not None and all(q is not r for r in out):
            out.append(q)
    return out


FILM_KEY = {id(q): n for n in FILM_NAMES for q in copies(n)}
GROUPS = {}   # one entry per independent parameter, the list of objects that carry it
for e in E_SWATH:
    for q in obj_at(e).varying_parameters():
        key = FILM_KEY.get(id(q), id(q))
        GROUPS.setdefault(key, copies(key) if key in FILM_NAMES else [q])
for n in ORDER:
    GROUPS.setdefault(n, copies(n))
PARAMS = list(GROUPS.values())
KIDX = {n: list(GROUPS).index(n) for n in ORDER}
KNOB_PARAMS = {n: GROUPS[n][0] for n in ORDER}
BLOCKS = [(e, pol) for e in E_SWATH for pol in POLS]
PTS = {b: _data_points(*b) for b in BLOCKS}


def ln_block(b):
    return np.log(rxr(PTS[b][0], obj_at(b[0]).model, b[1]))


J = {b: np.empty((len(PTS[b][0]), len(PARAMS))) for b in BLOCKS}
for j, group in enumerate(PARAMS):
    v0 = [float(q.value) for q in group]
    h = _step(group[0])
    try:
        for q in group:
            q.value = v0[0] + h
        up = {b: ln_block(b) for b in BLOCKS}
        for q in group:
            q.value = v0[0] - h
        dn = {b: ln_block(b) for b in BLOCKS}
    finally:
        for q, v in zip(group, v0):
            q.value = v
    for b in BLOCKS:
        J[b][:, j] = (up[b] - dn[b]) / (2 * h) * PTS[b][1] / PTS[b][2]

dead = [f"{n} at {e:g} eV {pol}" for n in ORDER for (e, pol) in BLOCKS
        if not np.any(J[e, pol][:, KIDX[n]])]
assert not dead, f"knob moves did not reach the reflectivity for {dead}"

RESID = {b: (np.log(PTS[b][1]) - ln_block(b)) * PTS[b][1] / PTS[b][2] for b in BLOCKS}
CHI2NU = {b: float(np.mean(RESID[b] ** 2)) for b in BLOCKS}
CHI2NU_ALL = float(sum(np.sum(r**2) for r in RESID.values())
                   / max(sum(len(r) for r in RESID.values()) - len(PARAMS), 1))
# Error model. In ln R the measured error is sigma_ln = y_err / y; the misfit is absorbed by a
# fractional floor f_E, sigma_eff^2 = sigma_ln^2 + f_E^2, solved per dataset for chi2_nu = 1.
# Points with the smallest error bars are inflated most, as a systematic floor should do.
from scipy.optimize import brentq

SIG_LN = {b: PTS[b][2] / PTS[b][1] for b in BLOCKS}


def error_floor(b):
    d2, s2 = (RESID[b] * SIG_LN[b]) ** 2, SIG_LN[b] ** 2
    if np.mean(d2 / s2) <= 1.0:
        return 0.0
    return brentq(lambda f: np.mean(d2 / (s2 + f * f)) - 1.0, 0.0, np.sqrt(d2.max()) + 1e-12)


F_FLOOR = {b: error_floor(b) for b in BLOCKS}
REWEIGHT = {b: (SIG_LN[b] / np.sqrt(SIG_LN[b] ** 2 + F_FLOOR[b] ** 2)
                if FISHER_SCALE == "error_model" else np.ones_like(SIG_LN[b])) for b in BLOCKS}
JW = {b: J[b] * REWEIGHT[b][:, None] for b in BLOCKS}      # Jacobian in units of sigma_eff
RW = {b: RESID[b] * REWEIGHT[b] for b in BLOCKS}           # residuals in units of sigma_eff
SCALE = {b: {"dataset": 1 / max(CHI2NU[b], 1.0), "global": 1 / max(CHI2NU_ALL, 1.0),
             None: 1.0, "error_model": 1.0}[FISHER_SCALE] for b in BLOCKS}
FISH = {b: SCALE[b] * JW[b].T @ JW[b] for b in BLOCKS}
print("fractional error floor f_E (%) per dataset",
      {f"{e:g}{pol}": round(100 * F_FLOOR[e, pol], 1) for e, pol in BLOCKS})
F_ALL = sum(FISH.values())


def knob_var(F):
    """Marginal variance of every knob from information matrix F, all parameters free.
    Parameters F carries no information on (another dataset's scale, say) are dropped
    first, since the pseudo-inverse would otherwise give them, and their partners, a
    false zero variance."""
    d = np.diag(F)
    keep = np.flatnonzero(d > 1e-12 * d.max())
    C = np.linalg.pinv(F[np.ix_(keep, keep)], rcond=1e-12)
    pos = {k: i for i, k in enumerate(keep)}
    return {n: (C[pos[KIDX[n]], pos[KIDX[n]]] if KIDX[n] in pos else np.inf) for n in ORDER}


DELTA = {n: abs(float(KNOB_PARAMS[n].value) - float(getattr(FILM, BULK[n]).value)) for n in ORDER}
VAR_ALL = knob_var(F_ALL)
SNR_JOINT = {n: DELTA[n] / np.sqrt(VAR_ALL[n]) for n in ORDER}
# Per-dataset Cramer-Rao bound on each knob, sigma_CR = (F_kk^(b))^(-1/2), the smallest
# standard error that dataset alone could give with every other parameter known, and the
# detectability |dtheta| / sigma_CR with dtheta = theta_pin - theta_bulk, the number of
# those standard errors between the pinned value and the bulk value it relaxes to.
F_KK = {b: {n: FISH[b][KIDX[n], KIDX[n]] for n in ORDER} for b in BLOCKS}
SIGMA_CR = {pol: {n: [F_KK[e, pol][n] ** -0.5 if F_KK[e, pol][n] > 0 else np.inf
                      for e in E_PANEL] for n in ORDER} for pol in POLS}
DETECT = {pol: {n: [DELTA[n] / s for s in SIGMA_CR[pol][n]] for n in ORDER} for pol in POLS}


def to_disp(n, dv):
    """A model-unit step at the best fit, in display units (deg or g cm^-3)."""
    f, t = BOUNDARY[n]["from_model"], BOUNDARY[n]["to_model"]
    return float(abs(f(t(BOUNDARY[n]["fitted"]) + dv) - BOUNDARY[n]["fitted"]))


display(pd.DataFrame({
    "best fit": {n: BOUNDARY[n]["fitted"] for n in ORDER},
    "bulk": {n: BOUNDARY[n]["from_model"](float(getattr(FILM, BULK[n]).value)) for n in ORDER},
    "joint sigma": {n: to_disp(n, np.sqrt(VAR_ALL[n])) for n in ORDER},
    "joint |dtheta|/sigma": SNR_JOINT}).round(3))
print(f"chi2_nu over all data {CHI2NU_ALL:.2f}; per dataset",
      {f"{e:g}{pol}": round(CHI2NU[e, pol], 1) for e, pol in BLOCKS})

# Stacking. In every panel (polarisation x knob) each energy is offset so that all of
# its swept curves sit GAP_DEC decades below all curves of the energy above, at every
# q >= Q_YMIN that both share. OFFS holds the multiplicative offset.
OFFS = {}
for pol in ROW_POL:
    for name in ORDER:
        prev_lo = None
        for e in E_LOWQ:
            a = np.log10(BSIM[name][e][pol] * Q**4)
            a[:, (Q < Q_YMIN) | (Q > QMEAS[e])] = np.nan
            shift = 0.0 if prev_lo is None else float(np.nanmin(prev_lo - np.nanmax(a, axis=0))) - GAP_DEC
            OFFS[pol, name, e] = 10.0**shift
            prev_lo = np.nanmin(a, axis=0) + shift

print("per-dataset Cramer-Rao bound sigma_CR (deg or g cm^-3)")
display(pd.DataFrame({(pol, e): [to_disp(n, SIGMA_CR[pol][n][k]) for n in ORDER]
                      for pol in ROW_POL for k, e in enumerate(E_PANEL)},
                     index=ORDER).round(3))
print("detectability |dtheta| / sigma_CR")
display(pd.DataFrame({(pol, e): [DETECT[pol][n][k] for n in ORDER]
                      for pol in ROW_POL for k, e in enumerate(E_PANEL)},
                     index=ORDER).round(1))

# Joint marginal separation |dtheta| / sigma, every parameter free, under three noise models.
#   "noise"     the FISHER_SCALE noise model, the expected Fisher information (as the bars)
#   "chi2_nu"   covariance inflated by the reduced chi^2 of the datasets used, misfit as white noise
#   "sandwich"  H^-1 (J^T diag(r^2) J) H^-1 with the fit residuals r, robust to a misspecified
#               noise model as long as the residuals are uncorrelated between points
def joint_var(blocks, model="noise"):
    Jb = np.vstack([JW[b] for b in blocks])     # same noise model as the bars (FISHER_SCALE)
    rb = np.concatenate([RW[b] for b in blocks])
    H = Jb.T @ Jb
    d = np.diag(H)
    keep = np.flatnonzero(d > 1e-12 * d.max())
    Hi = np.linalg.pinv(H[np.ix_(keep, keep)], rcond=1e-12)
    if model == "chi2_nu":
        C = Hi * max(float(np.sum(rb**2)) / max(len(rb) - len(keep), 1), 1.0)
    elif model == "sandwich":
        Jk = Jb[:, keep]
        C = Hi @ (Jk.T @ (Jk * rb[:, None] ** 2)) @ Hi
    else:
        C = Hi
    pos = {k: i for i, k in enumerate(keep)}
    return {n: (C[pos[KIDX[n]], pos[KIDX[n]]] if KIDX[n] in pos else np.inf) for n in ORDER}


CASES = {("all", "noise"): (BLOCKS, "noise"), ("all", "chi2_nu"): (BLOCKS, "chi2_nu"),
         ("all", "sandwich"): (BLOCKS, "sandwich"),
         ("p only", "noise"): ([b for b in BLOCKS if b[1] == "p"], "noise"),
         ("s only", "noise"): ([b for b in BLOCKS if b[1] == "s"], "noise")}
JOINT = {case: joint_var(*args) for case, args in CASES.items()}
print("joint marginal |dtheta| / sigma, every parameter free")
display(pd.DataFrame({case: {n: DELTA[n] / np.sqrt(v[n]) for n in ORDER}
                      for case, v in JOINT.items()}).round(1))
print("joint marginal sigma (deg or g cm^-3)")
display(pd.DataFrame({case: {n: to_disp(n, np.sqrt(v[n])) for n in ORDER}
                      for case, v in JOINT.items()}).round(3))


In [ ]:
import matplotlib as mpl
import matplotlib.ticker as mticker

ecol = dict(zip(E_PANEL, plt.get_cmap("plasma")(np.linspace(0.0, 0.85, len(E_PANEL)))))
QX_LO = 1.04 * max(QMEAS[e] for e in E_LOWQ)
LW = 0.7
# Text sizes are the set_plotting_defaults sizes times FONT_SCALE, applied to this figure only.
FONT_SCALE = 0.8
FONTS = {k: FONT_SCALE * mpl.font_manager.FontProperties(size=plt.rcParams[k]).get_size_in_points()
         for k in ("font.size", "axes.labelsize", "axes.titlesize", "xtick.labelsize",
                   "ytick.labelsize", "legend.fontsize")}
SMALL = FONTS["legend.fontsize"]                 # in-plot labels, the smallest default size
POL_NAME = {"p": "p-polarised", "s": "s-polarised"}
POL_COLOR = {"p": "#1f4e9c", "s": "#a3361f"}     # row label
ROW_GAP = 0.12                                   # blank row between the p and s blocks
CB_TICKS = {"alpha": [0, 30, 60, 90], "density": [0.5, 1.0, 1.5, 2.0]}   # colour-bar ticks
LABEL_SEP = 0.18                                 # minimum spacing of the energy labels, axes fraction

# y-range knobs, in decades of R q^4. "full" is the 283.7 eV axis, "stack" the offset axis.
Q_SHOW = {"full": 0.01, "stack": 0.01}     # drawn points below this q are ignored for the limits
FLOOR_PCT = {"full": 1.0, "stack": 0.0}    # bottom at this percentile of the drawn points (0 = deepest)
PAD_BOTTOM = {"full": 0.10, "stack": 0.15}
PAD_TOP = {"full": 0.05, "stack": 0.30}
YLIM_MANUAL = {}   # hard override per axis, e.g. {("s", "density_vac", "stack"): (1e-10, 1e-6)}


def set_ylim(ax, drawn, kind, pol, name):
    """Limits from the (q, Rq^4) pairs actually drawn on ``ax``."""
    if (pol, name, kind) in YLIM_MANUAL:
        return ax.set_ylim(*YLIM_MANUAL[pol, name, kind])
    y = np.log10(np.concatenate([v[(q >= Q_SHOW[kind]) & (v > 0)] for q, v in drawn]))
    ax.set_ylim(10 ** (np.percentile(y, FLOOR_PCT[kind]) - PAD_BOTTOM[kind]),
                10 ** (y.max() + PAD_TOP[kind]))


def no_yticklabels(ax):
    ax.yaxis.set_major_formatter(mticker.NullFormatter())
    ax.yaxis.set_minor_formatter(mticker.NullFormatter())


with plt.rc_context(FONTS):
    fig = plt.figure(figsize=(7.5, 4.5), dpi=FIG_DPI, constrained_layout=True)
    fig.get_layout_engine().set(h_pad=0.01, w_pad=0.01, hspace=0.015, wspace=0.02)
    gs = fig.add_gridspec(6, 6, width_ratios=[0.06, 1, 1, 1, 1, 1.05],
                          height_ratios=[0.04, 1, 1.15, ROW_GAP, 1, 1.15])
    ROW0 = {r: 1 + 3 * r for r in range(len(ROW_POL))}       # upper axis row of each block
    LETTER = lambda r, c: f"({'abcdefghij'[5 * r + c]})"    # one letter per panel, row by row
    bar_axes = []

    for c, name in enumerate(ORDER):
        spec = BOUNDARY[name]
        cm = plt.get_cmap(spec["cmap"])
        cols = cm(np.linspace(0, 1, len(spec["values"])))

        # column header, the swept knob and its colour scale with the published value ticked
        cax = fig.add_subplot(gs[0, c + 1])
        nrm = plt.Normalize(spec["values"].min(), spec["values"].max())
        cb = fig.colorbar(plt.cm.ScalarMappable(norm=nrm, cmap=cm), cax=cax,
                          orientation="horizontal")
        cb.ax.axvline(spec["fitted"], color="k", lw=1.2)
        cb.ax.minorticks_off()
        cb.set_ticks(CB_TICKS[name.split("_")[0]])
        cax.set_title(spec["label"])   # knob and its unit

        for r, pol in enumerate(ROW_POL):
            # upper axis, 283.7 eV over its full q range
            ax = fig.add_subplot(gs[ROW0[r], c + 1])
            drawn = []
            for col, curve in zip(cols, BSIM[name][E_HIGHQ][pol]):
                ax.semilogy(Q, curve * Q**4, color=col, lw=LW)
                drawn.append((Q, curve * Q**4))
            ax.set_xlim(0, Q_MAX)
            set_ylim(ax, drawn, "full", pol, name)
            ax.xaxis.set_major_locator(mticker.MultipleLocator(0.1))
            no_yticklabels(ax)
            ax.set_title(LETTER(r, c), loc="left")
            if c == 0:
                ax.set_ylabel(f"$Rq^4$\n{E_HIGHQ:g} eV")

            # lower axis, the other resonant energies over their measured q range, stacked
            ax = fig.add_subplot(gs[ROW0[r] + 1, c + 1])
            drawn, labels = [], []
            for e in E_LOWQ:
                m = Q <= QMEAS[e]
                off = OFFS[pol, name, e]
                for col, curve in zip(cols, BSIM[name][e][pol]):
                    ax.semilogy(Q[m], curve[m] * Q[m] ** 4 * off, color=col, lw=LW)
                    drawn.append((Q[m], curve[m] * Q[m] ** 4 * off))
                labels.append((e, np.log10(BSIM[name][e][pol][len(cols) // 2][m][-1]
                                           * QMEAS[e] ** 4 * off)))
            ax.set_xlim(0, QX_LO * 1.3)   # room on the right for the energy labels
            set_ylim(ax, drawn, "stack", pol, name)
            if c == 0:   # the stacking order is the same in every panel, so label it once
                lo, hi = np.log10(ax.get_ylim())
                yf = [0.93 + LABEL_SEP]   # top label no higher than 0.93 of the axis
                for e, ly in labels:   # top to bottom, nudged apart so no two labels overlap
                    yf.append(min((ly - lo) / (hi - lo), yf[-1] - LABEL_SEP))
                yf = np.array(yf[1:]) + max(0.0, 0.05 - yf[-1])   # lift the set if it ran off the bottom
                for (e, _), y in zip(labels, yf):
                    ax.text(0.985, y, f"{e:g}", transform=ax.transAxes, fontsize=SMALL,
                            va="center", ha="right", color=ecol[e])   # right-aligned inside the frame
            ax.xaxis.set_major_locator(mticker.MultipleLocator(0.04))
            ax.yaxis.set_major_locator(mticker.NullLocator())
            ax.yaxis.set_minor_locator(mticker.NullLocator())
            if c == 0:
                ax.set_ylabel("$Rq^4$\noffset")
            if r == len(ROW_POL) - 1:
                ax.set_xlabel(r"$q\ (\mathrm{\AA}^{-1})$")

    for r, pol in enumerate(ROW_POL):
        # polarisation named in the left margin
        lab = fig.add_subplot(gs[ROW0[r]: ROW0[r] + 2, 0])
        lab.axis("off")
        lab.text(0.5, 0.5, POL_NAME[pol], transform=lab.transAxes, rotation=90, ha="center",
                 va="center", fontsize=plt.rcParams["axes.labelsize"], color=POL_COLOR[pol])

        # detectability of each pinned state from each dataset alone, one bar per energy
        ax = fig.add_subplot(gs[ROW0[r]: ROW0[r] + 2, 5],
                             sharex=bar_axes[0] if bar_axes else None)
        bar_axes.append(ax)
        ax.grid(axis="x", visible=False)
        x = np.arange(len(ORDER))
        w = 0.84 / len(E_PANEL)
        # bars rise from 1, one Cramer-Rao standard error, so a bar's length is the
        # separation beyond the single-dataset noise floor; a value below 1 hangs down
        for k, e in enumerate(E_PANEL):
            v = np.array([DETECT[pol][n][k] for n in ORDER], dtype=float)
            ax.bar(x + (k - (len(E_PANEL) - 1) / 2) * w, v - 1.0, bottom=1.0, width=w,
                   color=ecol[e], edgecolor="none", label=f"{e:g} eV")
        ax.set_yscale("log")
        vals = np.array([DETECT[q][n] for q in ROW_POL for n in ORDER], dtype=float)
        vals = vals[np.isfinite(vals) & (vals > 0)]
        ax.set_ylim(min(1.0, 0.6 * vals.min()), 1.6 * vals.max())   # same range in (e) and (j)
        if vals.min() < 1.0:
            ax.axhline(1.0, color="k", lw=0.6)
        lo_y, hi_y = np.log10(ax.get_ylim())   # label 1, 2, 5 steps if fewer than 3 decades show
        n_dec = int(np.floor(hi_y) - np.ceil(lo_y)) + 1
        ax.yaxis.set_major_locator(mticker.LogLocator(subs=(1, 2, 5) if n_dec < 3 else (1,)))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
        ax.yaxis.set_minor_formatter(mticker.NullFormatter())
        ax.set_xticks(x, [SHORT[n] for n in ORDER])
        ax.tick_params(axis="x", which="both", length=0)
        ax.set_ylabel(r"$|\Delta\theta| / \sigma_\mathrm{CR}$")
        ax.set_title(LETTER(r, 4), loc="left")
        if r == 0:
            plt.setp(ax.get_xticklabels(), visible=False)
    lax = fig.add_subplot(gs[0, 5])
    lax.axis("off")
    lax.legend(*bar_axes[0].get_legend_handles_labels(), loc="center", ncols=3, frameon=False,
               handlelength=0.9, columnspacing=0.8, borderaxespad=0)
    fig.savefig("SI11.png", transparent=True, dpi=600)
    plt.show()
